## 1. Lý thuyết: Từ Word-level Tokenization sang Subword Tokenization

Trong các buổi lab trước (LAB 01 – LAB 04), việc tách từ dựa trên khoảng trắng hoặc từ điển tĩnh (Word-level Tokenization) gặp phải 2 vấn đề chí mạng:
1. **Bùng nổ kích thước từ vựng (Out-Of-Vocabulary - OOV):** Mọi từ chưa từng xuất hiện trong tập huấn luyện đều bị ánh xạ về token vô danh `[UNK]`, làm mất hoàn toàn ngữ nghĩa.
2. **Không chia sẻ hình thái học:** Các biến thể của cùng một gốc từ (*"play"*, *"plays"*, *"playing"*, *"player"*) được xem là các vector hoàn toàn độc lập.

**Giải pháp Subword Tokenization (WordPiece trong BERT):**
- Tách các từ phổ biến thành nguyên vẹn một token.
- Tách các từ hiếm hoặc từ phức thành các phân đoạn từ con (subwords), trong đó các phân đoạn phía sau được đánh dấu bằng tiền tố `##` (ví dụ: *"unbelievable"* $\to$ `['un', '##beli', '##eva', '##ble']`).
- Nhờ đó, từ điển chỉ cần kích thước cố định khoảng $30,000$ tokens nhưng có thể biểu diễn được hầu như mọi từ vựng tiếng Anh mà không bao giờ gặp lỗi OOV.


In [1]:
import os
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'

import sys
import math
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer

sys.stdout.reconfigure(encoding='utf-8')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', local_files_only=True)
print("BERT WordPiece Tokenizer loaded successfully.")
print(f"Vocabulary Size: {tokenizer.vocab_size:,} tokens")


BERT WordPiece Tokenizer loaded successfully.
Vocabulary Size: 30,522 tokens


## 2. Calculation Exercise 2: Phân tích Tokenization câu *"The cats are playing."*

Kiểm tra chi tiết chuỗi tokens, mã định danh token IDs, special tokens, attention mask và so sánh tương quan số lượng:
- Số từ tự nhiên (Words)
- Số token thực tế (Tokens)
- Số phân đoạn từ con (Subwords)


In [2]:
text_exercise = "The cats are playing."
encoded = tokenizer(text_exercise)

tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])
input_ids = encoded['input_ids']
attention_mask = encoded['attention_mask']

print("Input Text    :", text_exercise)
print("Token Sequence:", tokens)
print("Token IDs     :", input_ids)
print("Attention Mask:", attention_mask)

words_count = len(text_exercise.replace('.', '').split())
tokens_count = len(tokens)
subwords_count = sum(1 for t in tokens if t.startswith("##"))

summary_counts = [
    {"Metric": "Raw Words (Tách khoảng trắng)", "Count": words_count, "Details": "The, cats, are, playing"},
    {"Metric": "Tokens (Bao gồm Special & Punctuation)", "Count": tokens_count, "Details": "[CLS], the, cats, are, playing, ., [SEP]"},
    {"Metric": "Subwords (Phân đoạn có tiền tố ##)", "Count": subwords_count, "Details": "Các từ đều có sẵn trong từ điển 30,522"}
]

df_counts = pd.DataFrame(summary_counts)
print("\n" + df_counts.to_string(index=False))


Input Text    : The cats are playing.
Token Sequence: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
Token IDs     : [101, 1996, 8870, 2024, 2652, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1]

                                Metric  Count                                                  Details
             Raw Words (Tách khoảng trắng)      4                                  The, cats, are, playing
Tokens (Bao gồm Special & Punctuation)      7                  [CLS], the, cats, are, playing, ., [SEP]
        Subwords (Phân đoạn có tiền tố ##)      0                 Các từ đều có sẵn trong từ điển 30,522


### Biện luận Bài tập 2:
1. **Các Token đặc biệt (Special Tokens):**
   - `[CLS]` (ID = 101): Được chèn tự động ở đầu câu. Vector biểu diễn của token này tại tầng cuối cùng của BERT đóng vai trò là vector đại diện cho toàn bộ câu (Sequence Representation) trong các tác vụ phân loại.
   - `[SEP]` (ID = 102): Được chèn ở cuối câu để báo hiệu phân cách ranh giới câu.
2. **Attention Mask:**
   - Vector `[1, 1, 1, 1, 1, 1, 1]` tương ứng với 7 vị trí hợp lệ. Toàn bộ các vị trí này đều được tham gia tính toán trong ma trận Self-Attention ($s_{ij} = \frac{q_i^T k_j}{\sqrt{d_k}}$).


## 3. Khảo sát Subword Segmentation trên Từ ghép hiếm và Từ chưa biết (OOV)

Thử nghiệm tách từ trên các từ phức tạp có hậu tố, tiền tố hoặc từ mới sáng tạo:
- *"unbelievable"*
- *"transformer"*
- *"antigravity"*
- *"supercalifragilisticexpialidocious"*


In [3]:
test_words = ["unbelievable", "transformer", "antigravity", "supercalifragilisticexpialidocious"]
subword_results = []

for word in test_words:
    sub_tokens = tokenizer.tokenize(word)
    sub_ids = tokenizer.convert_tokens_to_ids(sub_tokens)
    subword_results.append({
        "Original Word": word,
        "Subword Tokens": sub_tokens,
        "Subword IDs": sub_ids,
        "Number of Subwords": len(sub_tokens)
    })

df_subwords = pd.DataFrame(subword_results)
for _, row in df_subwords.iterrows():
    print(f"Word: {row['Original Word']}")
    print(f"  -> Tokens: {row['Subword Tokens']}")
    print(f"  -> IDs   : {row['Subword IDs']}\n")


Word: unbelievable
  -> Tokens: ['un', '##beli', '##eva', '##ble']
  -> IDs   : [4895, 23727, 13735, 3965]

Word: transformer
  -> Tokens: ['transform', '##er']
  -> IDs   : [10938, 2121]

Word: antigravity
  -> Tokens: ['anti', '##gra', '##vity']
  -> IDs   : [3424, 18585, 17926]

Word: supercalifragilisticexpialidocious
  -> Tokens: ['super', '##cal', '##if', '##rag', '##ili', '##stic', '##exp', '##iali', '##do', '##cious']
  -> IDs   : [3565, 17822, 5971, 17354, 28318, 14210, 24083, 24878, 4372, 14013]


### Biện luận về Subword Tokenization:
- **Tiền tố `##`:** Biểu thị rằng phân đoạn từ con đó đứng liền sau một subword trước đó trong cùng một từ, không có khoảng trắng ngăn cách.
- **Tính ưu việt:** Ngay cả với một từ cực kỳ hiếm như *"supercalifragilisticexpialidocious"*, WordPiece vẫn tách thành 10 subwords quen thuộc và bảo toàn thông tin thay vì biến thành `[UNK]` như các tokenizer truyền thống.


## 4. Khảo sát Padding & Attention Mask trên Batch không đều

Khi gom nhiều câu có độ dài khác nhau vào một Mini-Batch, các câu ngắn hơn phải được chèn token đệm `[PAD]` (ID = 0) để tạo thành một Tensor chữ nhật đồng nhất.
Attention Mask đóng vai trò là tấm khiên ngăn không cho các token `[PAD]` tham gia vào phân phối Self-Attention.


In [4]:
batch_sentences = [
    "I like cats.",
    "Natural language processing with transformers is amazing."
]

batch_encoded = tokenizer(
    batch_sentences,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

print("Batch Input IDs Shape      :", batch_encoded['input_ids'].shape)
print("Batch Attention Mask Shape :", batch_encoded['attention_mask'].shape)
print("\nSentence 1 Input IDs       :", batch_encoded['input_ids'][0].tolist())
print("Sentence 1 Attention Mask  :", batch_encoded['attention_mask'][0].tolist())
print("Sentence 1 Tokens          :", tokenizer.convert_ids_to_tokens(batch_encoded['input_ids'][0]))
print("\nSentence 2 Input IDs       :", batch_encoded['input_ids'][1].tolist())
print("Sentence 2 Attention Mask  :", batch_encoded['attention_mask'][1].tolist())
print("Sentence 2 Tokens          :", tokenizer.convert_ids_to_tokens(batch_encoded['input_ids'][1]))


Batch Input IDs Shape      : torch.Size([2, 9])
Batch Attention Mask Shape : torch.Size([2, 9])

Sentence 1 Input IDs       : [101, 1045, 2066, 8870, 1012, 102, 0, 0, 0]
Sentence 1 Attention Mask  : [1, 1, 1, 1, 1, 1, 0, 0, 0]
Sentence 1 Tokens          : ['[CLS]', 'i', 'like', 'cats', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]']

Sentence 2 Input IDs       : [101, 3019, 2653, 6364, 2007, 19081, 2003, 6429, 102]
Sentence 2 Attention Mask  : [1, 1, 1, 1, 1, 1, 1, 1, 1]
Sentence 2 Tokens          : ['[CLS]', 'natural', 'language', 'processing', 'with', 'transformers', 'is', 'amazing', '[SEP]']


## 5. Calculation Exercise 1: Mô phỏng Masked Language Modeling (MLM) & Loss

Trong quá trình tiền huấn luyện BERT, khoảng $15\%$ số tokens trong văn bản được chọn ngẫu nhiên để áp dụng cơ chế Masking (trong đó $80\%$ biến thành `[MASK]`, $10\%$ giữ nguyên và $10\%$ thay bằng từ ngẫu nhiên). Mô hình phải tối thiểu hóa Cross-Entropy Loss trên các vị trí bị che:
$$\mathcal{L}_{MLM} = -\sum_{i \in \mathcal{M}} \ln P(y_i | x)$$

Thực nghiệm đo lường sự thay đổi của hàm mất mát khi xác suất dự đoán thay đổi ($P \in [0.01, 0.99]$):


In [5]:
prob_values = [0.01, 0.10, 0.30, 0.50, 0.70, 0.90, 0.95, 0.99]
losses_data = []

for p in prob_values:
    loss_val = -math.log(p)
    losses_data.append({
        "P(Correct Token)": f"{p:.2f}",
        "Cross-Entropy Loss": f"{loss_val:.4f}",
        "Penalty Ratio vs (P=0.70)": f"{loss_val / (-math.log(0.70)):.2f}x"
    })

df_mlm_loss = pd.DataFrame(losses_data)
print(df_mlm_loss.to_string(index=False))

p_plot = np.linspace(0.01, 0.99, 100)
loss_plot = -np.log(p_plot)

plt.figure(figsize=(8, 4), dpi=100)
plt.plot(p_plot, loss_plot, color="#d62728", linewidth=2.5, label=r"$\mathcal{L} = -\ln(P)$")
plt.scatter([0.70], [-math.log(0.70)], color="blue", s=80, zorder=5, label=r"Exercise Point: P=0.70 ($\mathcal{L}=0.3567$)")
plt.scatter([0.95], [-math.log(0.95)], color="green", s=80, zorder=5, label=r"High Confidence: P=0.95 ($\mathcal{L}=0.0513$)")
plt.scatter([0.01], [-math.log(0.01)], color="red", s=80, zorder=5, label=r"Severe Mistake: P=0.01 ($\mathcal{L}=4.6052$)")

plt.title("Cross-Entropy Loss Curve for Masked Language Modeling", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Probability for Correct Token P(y|x)", fontsize=11, fontweight="bold")
plt.ylabel("Loss Value", fontsize=11, fontweight="bold")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


P(Correct Token) Cross-Entropy Loss Penalty Ratio vs (P=0.70)
            0.01             4.6052                    12.91x
            0.10             2.3026                     6.46x
            0.30             1.2040                     3.38x
            0.50             0.6931                     1.94x
            0.70             0.3567                     1.00x
            0.90             0.1054                     0.30x
            0.95             0.0513                     0.14x
            0.99             0.0101                     0.03x


## 6. Tổng kết bài học về Tokenization

1. **Subword Tokenization (WordPiece)** giải quyết triệt để vấn đề Out-Of-Vocabulary (OOV) bằng cách phân rã các từ lạ thành các phân đoạn từ con có tiền tố `##`.
2. **Special Tokens (`[CLS]`, `[SEP]`, `[PAD]`)** cung cấp cấu trúc ngữ pháp và điểm neo đại diện toàn câu cho mô hình Transformer.
3. **Attention Mask** đảm bảo các token chèn đệm `[PAD]` không làm sai lệch luồng thông tin trong mạng Self-Attention.
4. **Hàm mất mát MLM** phạt theo hàm logarit dốc đứng đối với các dự đoán sai lệch, thúc đẩy mô hình học được phân phối xác suất ngữ nghĩa sâu sắc của ngôn ngữ tự nhiên.
